## Topic: Persistence In LangGraph

### Agenda

- 0. Til we Learn

- 1. Introduction of Persistence

- 2. Why Persistence Matters in AI Agents

- 3. Checkpointers in Persistence

- 4. Thread_ID in Persistence

- 5. Key Takeaways

## 0. Til we Learn

- 1. Concept of Graph:
    - how to decompose high level goal into set of task or sub tasks
    - and represent the set of tasks into graph format.
    - whereas, each nodes are refers to the specific task or sub-tasks.
    - and the edges refers to the execution order of task or sub-tasks 


- 2. Concept of State:
    - Before Define a Graph, we must define a Graph state.
    - State are represented, the importance data that are need all task or sub-tasks to execute their operation.
    - state is nothing but a dictionary.
    - All node of our graph(workflow) are access the state data and modify the data.

- Now the Problems:
    - when we provide the input by using the invoke() method, 
        - for each new input(query) the previous input value(query) are automatically updates or replace.  

        - for one execution is complete, the state value are erase automatically.

        - so that, after one execution of our workflow, we can't access the state values.

        - This is the core behavior of LangGraph.

- Solution:
    - To solve this **state erase problem**, we use the **Persistence** to store the **state** value which can be temporary (RAM) or permanently (DB).

### 1. Introduction of Persistence

- Definition:
    - Persistence in LanGraph refers to the ability to **save and restore** the **state** of a workflow over time.


    - Persistence not only store the final_state value but also store the all intermediate state/nodes/steps values.


- Key Rule of Thump:
    - Persistence in LangGraph is the ability to automatically save and restore the full state of our graph at every step (node execution). Instead of losing all context the moment a graph finishes running, the state is written to a storage layer called a Checkpointer, and can be loaded back later using a thread ID.

""" 
Without Persistence (Stateless):
================================
  User: "My name is Kz"
  Graph runs → responds → STATE IS DESTROYED 
  User: "What is my name?"
  Graph runs → "I don't know." (no memory!)

With Persistence (Stateful):
================================
  User: "My name is Kz"
  Graph runs → responds → STATE IS SAVED  (checkpoint created)
  User: "What is my name?"
  Graph runs → loads checkpoint → "Your name is KZ!" (remembers!)

"""

### 2. Why Persistence Matters in AI Agents

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           WHY PERSISTENCE MATTERS                           │
│                                                             │
│  1. MULTI-TURN CONVERSATIONS                                │
│     Users expect the agent to remember what was said        │
│     5 messages ago, 50 messages ago, or 5 days ago.         │
│     Without persistence, every turn is a blank slate.       │
│                                                             │
│  2. HUMAN-IN-THE-LOOP WORKFLOWS                             │
│     The graph pauses and waits for human approval.          │
│     The state MUST survive the pause (minutes or hours).    │
│     Persistence keeps the graph "frozen" until resumed.     │
│                                                             │
│  3. FAULT TOLERANCE                                         │
│     Your server crashes mid-execution.                      │
│     Without persistence: all progress lost.                 │
│     With persistence: resume from the last checkpoint.      │
│                                                             │
│  4. DEBUGGING & OBSERVABILITY                               │
│     "Why did the agent give that wrong answer?"             │
│     Persistence lets you inspect EVERY intermediate state   │
│     to trace exactly where things went wrong.               │
│                                                             │
│  5. TIME TRAVEL & REPLAY                                    │
│     Go back to any previous checkpoint and branch off       │
│     in a new direction. Invaluable for experimentation      │
│     and correcting mistakes retroactively.                  │
│                                                             │
│  6. LONG-RUNNING AGENT TASKS                                │
│     An agent that researches, writes, and revises over      │
│     hours needs its state to survive between sessions.      │
│     Persistence makes this possible.                        │
└─────────────────────────────────────────────────────────────┘

"""

### 3. Checkpointers in Persistence

- Definition: 
    - A Checkpointer is a storage backend that LangGraph calls automatically after every node execution. You don't write save/load logic yourself — the checkpointer handles it transparently.


    - Persistence are implementation in LangGraph by using Checkpointers.

    - Checkpointers define the checkpointers on the Graph, For each super steps has one checkpointer.

    - When the graph is execute the checkpointer store the state value of each super steps.


In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           STATELESS vs STATEFUL GRAPHS                      │
│                                                             │
│  STATELESS (No Checkpointer):                               │
│  ────────────────────────────                               │
│  graph = builder.compile()   # No checkpointer!             │
│                                                             │
│  Invoke 1: {"messages": ["Hi, I'm Kz"]}                     │
│    → Graph processes → Returns response → STATE GONE        │
│                                                             │
│  Invoke 2: {"messages": ["What's my name?"]}                │
│    → Graph processes → "I don't know" → STATE GONE          │
│                                                             │
│  Each invoke is completely independent.                     │
│  The graph has amnesia between calls.                       │
│                                                             │
│  STATEFUL (With Checkpointer):                              │
│  ────────────────────────────                               │
│  memory = MemorySaver()                                     │
│  graph = builder.compile(checkpointer=memory)               │
│                                                             │
│  Invoke 1: {"messages": ["Hi, I'm Kz"]}                     │
│    → Graph processes → Returns response → STATE SAVED       │
│      (checkpoint_id: abc123)                                │
│                                                             │
│  Invoke 2: {"messages": ["What's my name?"]}                │
│    → Graph loads checkpoint → Sees "Kz" → "Kz!"             │
│      (checkpoint_id: def456)                                │
│                                                             │
│  Each invoke builds on the previous one.                    │
│  The graph has memory across calls.                         │
│                                                             │
│  KEY DIFFERENCE:                                            │
│  ────────────────                                           │
│  Stateless: compile()              → Amnesia                │
│  Stateful:  compile(checkpointer=) → Memory                 │
└─────────────────────────────────────────────────────────────┘


"""

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           WHAT'S INSIDE A CHECKPOINT?                       │
│                                                             │
│  A checkpoint is a snapshot containing:                     │
│                                                             │
│  1. VALUES                                                  │
│     The full state at that moment.                          │
│     Example: {"messages": [msg1, msg2, msg3]}               │
│                                                             │
│  2. NEXT                                                    │
│     The list of nodes scheduled to run next.                │
│     Empty [] if the graph has finished.                     │
│     Example: ["tools"] (if the agent needs to call a tool)  │
│                                                             │
│  3. CONFIG                                                  │
│     The configuration at that point, including:             │
│     - thread_id                                             │
│     - checkpoint_id (unique per save)                       │
│     - checkpoint_ns (namespace for subgraphs)               │
│                                                             │
│  4. METADATA                                                │
│     Extra info about the step:                              │
│     - step: which step number                               │
│     - source: "loop" | "input" | "update"                   │
│     - writes: what was written in this step                 │
│                                                             │
│  5. CREATED_AT                                              │
│     Timestamp of when the checkpoint was saved.             │
└─────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" - Available Checkpointers (Memory, SQLite, Postgres): 
    =======================================================


┌─────────────────────────────────────────────────────────────┐
│           CHECKPOINTER COMPARISON                           │
│                                                             │
│  ┌──────────────┬───────────┬───────────┬───────────────┐   │
│  │ Feature      │ Memory    │ SQLite    │ PostgreSQL    │   │
│  ├──────────────┼───────────┼───────────┼───────────────┤   │
│  │ Storage      │ RAM       │ File      │ Database      │   │
│  │ Survives     │           │           │               │   │
│  │  restart?    │  No       │  Yes      │  Yes          │   │
│  │ Production   │  No       │  Maybe    │  Yes          │   │
│  │  ready?      │           │ (small)   │               │   │
│  │ Concurrency  │  Single   │  Limited  │  High         │   │
│  │ Setup        │ Trivial   │ Easy      │ Moderate      │   │
│  │ Dependencies │ None      │ sqlite3   │ psycopg       │   │
│  │ Best for     │ Dev/Test  │ Prototypes│ Production    │   │
│  └──────────────┴───────────┴───────────┴───────────────┘   │
│                                                             │
│  RECOMMENDATION:                                            │
│  ───────────────                                            │
│  • Learning & testing    → MemorySaver                      │
│  • Single-user apps      → SqliteSaver                      │
│  • Multi-user production → PostgresSaver                    │
└─────────────────────────────────────────────────────────────┘

"""

### 4. Thread_ID in Persistence

- Definition:
    - The thread_id is the single most important concept in LangGraph persistence. It acts as the conversation identifier — each unique thread_id gets its own isolated chain of checkpoints.



In [ ]:
""" 
How Thread IDs Work:

  Thread ID: "user_alice"
  ─────────────────────────
  Checkpoint 1: ["Hi, I'm Alice"]
  Checkpoint 2: ["Hi, I'm Alice", "Hello Alice!"]
  Checkpoint 3: ["Hi, I'm Alice", "Hello Alice!", "I like Python"]
  Checkpoint 4: [..., "I like Python", "Great choice!"]
  (All linked together under "user_alice")

  Thread ID: "user_bob"
  ─────────────────────────
  Checkpoint 1: ["Hey, call me Bob"]
  Checkpoint 2: ["Hey, call me Bob", "Hi Bob!"]
  (Completely separate from Alice's conversation!)

  Thread ID: "support_ticket_4521"
  ─────────────────────────────────
  Checkpoint 1: ["My order is broken"]
  Checkpoint 2: ["My order is broken", "Sorry to hear that..."]
  (A customer support thread, isolated from both users)


"""

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           THREAD ID RULES                                   │
│                                                             │
│  1. SAME thread_id = SAME conversation                      │
│     State accumulates. The graph remembers everything.      │
│                                                             │
│  2. DIFFERENT thread_id = DIFFERENT conversation            │
│     Completely isolated. No cross-contamination.            │
│                                                             │
│  3. thread_id is YOUR choice                                │
│     Use user IDs, session IDs, ticket numbers, etc.         │
│     Example: "user_123", "session_abc", "ticket_4521"       │
│                                                             │
│  4. thread_id goes in the config                            │
│     config = {"configurable": {"thread_id": "my_thread"}}   │
│                                                             │
│  5. Every invoke with the same thread_id APPENDS            │
│     New messages are merged into the existing state.        │
└─────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           STATE GROWTH BY FIELD TYPE                        │
│                                                             │
│  With Reducer (e.g., add_messages):                         │
│    New values are APPENDED/MERGED with old values.          │
│    messages: [old...] + [new...] → [old..., new...]         │
│    Growth: UNBOUNDED (keeps growing)                        │
│                                                             │
│  Without Reducer (plain field):                             │
│    New values OVERWRITE old values.                         │
│    status: "running" → "done" (replaced, not appended)      │
│    Growth: FIXED (always one value)                         │
│                                                             │
│  With Custom Reducer:                                       │
│    You control the merge logic.                             │
│    Example: keep only last 10 messages.                     │
│    Growth: BOUNDED (you decide)                             │
└─────────────────────────────────────────────────────────────┘

"""

### 5. Key Takeaways

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           PERSISTENCE — KEY TAKEAWAYS                       │
│                                                             │
│  1. CHECKPOINTER = Automatic state saving                   │
│     Add checkpointer= to compile() and every node           │
│     execution is saved automatically.                       │
│                                                             │
│  2. THREAD_ID = Conversation identity                       │
│     Same thread_id → same conversation (state accumulates)  │
│     Different thread_id → different conversation (isolated) │
│                                                             │
│  3. get_state() = Inspect the present                       │
│     See the current state and what node runs next.          │
│                                                             │
│  4. get_state_history() = Inspect the past                  │
│     Walk through every checkpoint for full audit trail.     │
│                                                             │
│  5. Time travel = Replay from any checkpoint                │
│     Rewind to a previous state and branch in a new          │
│     direction. Original timeline is untouched.              │
│                                                             │
│  6. update_state() = Manual state modification              │
│     Inject messages, correct mistakes, override values.     │
│     Creates a new checkpoint (doesn't modify old ones).     │
│                                                             │
│  7. WATCH OUT FOR STATE BLOAT                               │
│     Messages accumulate forever with add_messages.          │
│     Use trim_messages, summarization, or session-based      │
│     thread_ids to keep state manageable.                    │
│                                                             │
│  8. CHOOSE THE RIGHT CHECKPOINTER                           │
│     MemorySaver   → development & testing                   │
│     SqliteSaver   → lightweight persistence                 │
│     PostgresSaver → production at scale                     │
│                                                             │
│  9. PERSISTENCE ENABLES HUMAN-IN-THE-LOOP                   │
│     The graph can pause for hours and resume seamlessly     │
│     because the state is durably saved.                     │
│                                                             │
│  10. THE ONE-LINE DIFFERENCE                                │
│      Stateless: graph = builder.compile()                   │
│      Stateful:  graph = builder.compile(checkpointer=mem)   │
│      That single argument unlocks all of the above.         │
└─────────────────────────────────────────────────────────────┘


"""